# Notebook 05: Final Evaluation (Baseline vs FinBERT)

## English
This notebook compares the baseline TF-IDF + Logistic Regression model against the fine-tuned FinBERT model on the held-out test set.
It loads saved artifacts, generates predictions, reports classification metrics, and stores a comparison table for reporting.

**Goals:**
- Reproduce test performance for both models.
- Quantify the improvement from fine-tuning.
- Export a compact metrics table for the report.

## Espanol
Este notebook compara el baseline TF-IDF + Logistic Regression contra el modelo FinBERT fine-tuned en el set de test.
Carga artefactos guardados, genera predicciones, reporta metricas de clasificacion y guarda una tabla comparativa.

**Objetivos:**
- Reproducir el rendimiento en test de ambos modelos.
- Cuantificar la mejora por fine-tuning.
- Exportar una tabla compacta de metricas para el reporte.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

# 1. Cargar datos de test
df_test = pd.read_csv("../data/processed/test.csv")
X_test = df_test["sentence"]
y_test = df_test["label"]

In [ ]:
# 2. Cargar Baseline (Logistic Regression)
print("Cargando Baseline...")
baseline_model = joblib.load("../models/baseline/logistic_regression_model.joblib")
vectorizer = joblib.load("../models/baseline/tfidf_vectorizer.joblib")

# Predicciones Baseline
X_test_tfidf = vectorizer.transform(X_test)
y_pred_baseline = baseline_model.predict(X_test_tfidf)

Cargando Baseline...


In [ ]:
# 3. Cargar Modelo Final (FinBERT)
print("Cargando FinBERT...")
device = 0 if torch.cuda.is_available() else -1

MODEL_DIR = "../models/finbert_final"

pipe = pipeline(
    "text-classification", model=MODEL_DIR, tokenizer=MODEL_DIR, device=device
)

Cargando FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# 3.1 Predicciones FinBERT
print("Prediciendo con FinBERT...")

batch_size = 16
y_pred_finbert = []

for i in range(0, len(X_test), batch_size):
    batch = X_test.iloc[i : i + batch_size].tolist()
    results = pipe(batch, truncation=True, max_length=512)
    y_pred_finbert.extend([r["label"] for r in results])

print(f"✅ Predicciones FinBERT completadas: {len(y_pred_finbert)} ejemplos")

Prediciendo con FinBERT...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Predicciones FinBERT completadas: 726 ejemplos


In [ ]:
# 4. Comparación de Métricas
print("\n--- REPORTE BASELINE (Logistic Regression) ---")
print(classification_report(y_test, y_pred_baseline))

print("\n--- REPORTE FINAL (FinBERT) ---")
print(classification_report(y_test, y_pred_finbert))


--- REPORTE BASELINE (Logistic Regression) ---
              precision    recall  f1-score   support

    negative       0.63      0.69      0.66        91
     neutral       0.84      0.81      0.82       431
    positive       0.64      0.67      0.66       204

    accuracy                           0.75       726
   macro avg       0.70      0.72      0.71       726
weighted avg       0.76      0.75      0.76       726


--- REPORTE FINAL (FinBERT) ---
              precision    recall  f1-score   support

    negative       0.90      0.82      0.86        91
     neutral       0.93      0.96      0.95       431
    positive       0.93      0.90      0.92       204

    accuracy                           0.93       726
   macro avg       0.92      0.90      0.91       726
weighted avg       0.93      0.93      0.93       726



In [ ]:
# 5. Tabla Comparativa
from sklearn.metrics import f1_score, accuracy_score

metrics = {
    "Model": ["Baseline (LogReg)", "FinBERT"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_finbert),
    ],
    "F1-Macro": [
        f1_score(y_test, y_pred_baseline, average="macro"),
        f1_score(y_test, y_pred_finbert, average="macro"),
    ],
}

df_compare = pd.DataFrame(metrics)
print("\nComparación Final:")
print(df_compare)


Comparación Final:
               Model  Accuracy  F1-Macro
0  Baseline (LogReg)  0.753444  0.712601
1            FinBERT  0.926997  0.907566


In [ ]:
# 6. Guardar resultados para el reporte
import os

output_dir = "../reports/metrics"
os.makedirs(output_dir, exist_ok=True)
df_compare.to_csv("../reports/metrics/final_comparison.csv", index=False)

# Results Snapshot

## English
- Baseline accuracy is ~0.75, while FinBERT reaches ~0.88 on the same test set.
- Macro F1 improves to ~0.86 with FinBERT, indicating better balance across classes.
- Final metrics are exported to `reports/metrics/final_comparison.csv` for reporting.

## Espanol
- El baseline logra accuracy ~0.75, mientras FinBERT alcanza ~0.88 en el mismo test.
- El F1 macro sube a ~0.86 con FinBERT, indicando mejor balance entre clases.
- Las metricas finales se exportan a `reports/metrics/final_comparison.csv` para el reporte.

# Conclusions

## English
- Fine-tuning delivers a clear lift over the baseline across accuracy and macro F1, confirming the value of domain-specific language modeling.
- The comparison table is persisted for traceable reporting and downstream dashboards.
- For production, monitor class drift and periodically re-evaluate the model on fresh financial text.

## Espanol
- El fine-tuning ofrece una mejora clara sobre el baseline en accuracy y F1 macro, validando el valor de un modelo de lenguaje financiero.
- La tabla comparativa queda guardada para un reporte trazable y uso en dashboards.
- Para produccion, monitorear drift de clases y re-evaluar periodicamente con texto financiero reciente.